# V2G capacity experiments

Open this notebook from the project folder and run all cells. The capacity data are included in `study-data.zip`. Each run saves its results under `runs/`.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

cplex_bin = os.environ.get("CPLEX_BIN")  # Set to your CPLEX Studio executable to repeat the solver check.
project = Path.cwd()
data_archive = project / "study-data.zip"

if not (project / "replication/run_focused_study.py").is_file():
    raise FileNotFoundError("Open this notebook from the v2g-capacity project folder.")
if not data_archive.is_file():
    raise FileNotFoundError(f"Place the study data archive here or edit data_archive: {data_archive}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(project / "requirements-capacity.txt")], check=True)

## Prepare the run

The archive supplies the study inputs and reference results. The scripts come from this checkout. Each experiment writes its own log.

In [ ]:
from datetime import datetime
import json
import os
from pathlib import PurePosixPath
import shutil
import tempfile
from zipfile import ZipFile

from replication.run_focused_study import INPUT_CHECKS, OPTIMIZATION_STEPS, OUTPUT_CHECKS

runs = project / "runs"
runs.mkdir(exist_ok=True)
work = Path(tempfile.mkdtemp(prefix=datetime.now().strftime("%Y%m%d-%H%M%S-"), dir=runs))
scripts = work / "replication"
shutil.copytree(project / "replication", scripts, ignore=shutil.ignore_patterns("__pycache__"))
(work / "outputs").mkdir()
logs = work / "logs"
logs.mkdir()
data_folders = {"capacity_checks", "cohort_reconstruction", "independent_validation", "sensitivity_checks"}

with ZipFile(data_archive) as archive:
    for entry in archive.infolist():
        path = PurePosixPath(entry.filename)
        if (len(path.parts) < 3 or path.parts[0] != "replication"
                or path.parts[1] not in data_folders or path.suffix not in {".json", ".csv", ".npz"}):
            continue
        if any(part in {".", ".."} or ":" in part or "\\" in part for part in path.parts):
            raise ValueError(f"Invalid archive path: {path}")
        target = work.joinpath(*path.parts)
        if target.exists():
            raise ValueError(f"Archive overlaps a project file: {path}")
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(entry) as source, target.open("xb") as dest:
            shutil.copyfileobj(source, dest)

validation = scripts / "independent_validation"
fresh = validation / "recomputed" / "full-study"

def run_step(script, *args):
    log = logs / f"{Path(script).stem}.log"
    print(f"Running {script}", flush=True)
    with log.open("w", encoding="utf-8") as stream:
        result = subprocess.run(
            [sys.executable, "-u", scripts / script, *args], cwd=work,
            stdout=stream, stderr=subprocess.STDOUT,
            env={**os.environ, "PYTHONUTF8": "1", "MPLBACKEND": "Agg"},
        )
    if result.returncode:
        print(log.read_text(encoding="utf-8", errors="replace")[-4000:])
        raise RuntimeError(f"{script} failed. See {log}")

print(f"Run folder: {work}")

## Reconstruct the inputs

Check the selected Caltech sessions and the generated fleet inputs against the study records.

In [ ]:
for step in INPUT_CHECKS:
    run_step(*step)

## Run the optimization experiments

Recompute the development fleet frontiers, timing and energy sensitivity, and the 156-fleet evaluation. The contract and chronology experiments use separate analytical cases. The fresh evaluation compares every capacity and solver status with its archived reference.

In [ ]:
for script, *args in OPTIMIZATION_STEPS:
    if script == "independent_validation/recompute_validation.py":
        args = ["--output", str(fresh)]
    run_step(script, *args)

if cplex_bin:
    run_step("independent_validation/check_cplex.py", "--cplex", cplex_bin,
             "--data", str(data_archive), "--output", str(work / "outputs/cplex_frontiers.csv"))

## Check the implementations and rebuild the figures

The independent checks compare capacities and certificate calculations against alternative formulations. Publication figures and tables use the archived evaluation after the fresh comparison passes.

In [ ]:
run_step("independent_validation/check_weighted_capacity.py")
run_step("independent_validation/audit_generalized_implementation.py")
for step in OUTPUT_CHECKS:
    run_step(*step)
run_step("independent_validation/summarize_savings.py")
print((logs / "summarize_savings.log").read_text(encoding="utf-8"))

## Results

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

comparison = read_json(fresh / "RECOMPUTATION_CHECK.json")
cohorts = read_json(fresh / "validation_summary.json")["cohorts"]
checks = read_json(validation / "generalized_implementation_audit.json")
sensitivity = read_json(scripts / "sensitivity_checks/sensitivity_summary.json")

print(f"Fresh evaluation: {comparison['compared_frontier_values']:,} capacities compared")
print(f"Maximum difference from the archive: {comparison['maximum_capacity_difference_kw']:.3g} kW")
for cohort, row in cohorts.items():
    print(f"{cohort}: {row['contexts']} fleets, {row['frontier_calls']} optimization runs")
print(f"Sensitivity: {sensitivity['optimal_sensitivity_lp_calls']} optimal, "
      f"{sensitivity['infeasible_sensitivity_lp_calls']} infeasible")
print(f"Independent audit: {checks['status']} "
      f"({checks['capacity_comparison_pairs']} capacity and {checks['credit_comparisons']} credit comparisons)")
print(f"Tables and figures: {work / 'outputs'}")
print(f"Fresh results: {fresh}")